# Sparse Hyperparameter Tuning (Colab Ready)

This notebook finds a strong sparse-attention configuration using a **two-stage search**:

1. **Stage 1 (coarse sweep):** many short runs to filter bad regions quickly.
2. **Stage 2 (confirm):** rerun top candidates with larger token budget and more seeds.

It is designed for your current Orion setup:
- sparse backend only (`attention.backend=sparse`)
- flex-only (`attention.sparse_impl=flex`)
- fixed window (`window_size=W`) and tuned degree (`expander_degree=d`)

The final selection uses a weighted score (quality > speed > VRAM), then writes the best config YAML.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()

if IN_COLAB and not REPO_ROOT.exists():
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dev.txt",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "seaborn", "matplotlib"], check=True
)
print("Dependencies installed.")

In [ ]:
import itertools
import json
import math
import os
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

from orion.data.shakespeare import load_tiny_shakespeare
from orion.experiments import evaluate_val_ppl

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

In [ ]:
# =============================
# User-tunable search settings
# =============================
EXPERIMENT_NAME = "sparse_hparam_tuning"

# Keep window fixed; tune d up to 256.
WINDOW_SIZE = 64
EXPANDER_DEGREES = [8, 16, 32, 64, 128, 256]

# Tune optimizer + sparse stability controls.
LR_GRID = [3e-4, 2e-4]
QK_NORM_OPTIONS = [False, True]
ORTHO_INIT_OPTIONS = [False]
SPECTRAL_NORM_OPTIONS = [False]

# Tune for one target seq_len first (recommended for practical budget).
SEQ_LEN = 1024
BATCH_BY_SEQ = {256: 8, 512: 4, 1024: 2, 2048: 1, 4096: 1}

# Stage budgets
STAGE1_SEEDS = [123]
STAGE2_SEEDS = [123, 456]
STAGE1_TRAIN_TOKENS = 1_500_000
STAGE2_TRAIN_TOKENS = 4_000_000
TOP_K = 4

# Optional cap if you want faster iteration. Set to None for full grid.
MAX_STAGE1_RUNS = None

# Model shape for tuning runs
MODEL_CFG = {
    "name": "orion",
    "d_model": 256,
    "n_layers": 4,
    "n_heads": 4,
    "mlp_mult": 4,
}

VAL_EVAL_BATCHES_STAGE2 = 40

assert SEQ_LEN in BATCH_BY_SEQ, f"Missing batch for seq_len={SEQ_LEN}"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for sparse_impl='flex'. In Colab, enable GPU runtime.")

DEVICE = "cuda"
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path("runs") / f"{EXPERIMENT_NAME}_{TIMESTAMP}"
CFG_ROOT = RUN_ROOT / "generated_configs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CFG_ROOT.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Run root:", RUN_ROOT)

# Ensure tinyshakespeare exists before launching runs.
load_tiny_shakespeare("data")
print("Dataset ready.")

In [ ]:
def steps_for_token_budget(seq_len: int, batch_size: int, token_budget: int) -> int:
    return max(1, math.ceil(token_budget / (seq_len * batch_size)))


def build_sparse_cfg(
    *,
    out_dir: Path,
    seed: int,
    seq_len: int,
    batch_size: int,
    lr: float,
    window_size: int,
    expander_degree: int,
    qk_norm: bool,
    ortho_init: bool,
    spectral_norm: bool,
    train_tokens_target: int,
) -> dict:
    steps = steps_for_token_budget(seq_len, batch_size, train_tokens_target)
    return {
        "run": {
            "out_dir": str(out_dir),
            "seed": int(seed),
            "steps": int(steps),
            "log_every": 10,
            "save_every": int(steps),
        },
        "data": {
            "dataset": "tinyshakespeare",
            "root": "data",
            "seq_len": int(seq_len),
            "batch_size": int(batch_size),
        },
        "model": MODEL_CFG.copy(),
        "attention": {
            "backend": "sparse",
            "window_size": int(window_size),
            "expander_degree": int(expander_degree),
            "sparse_impl": "flex",
            "sparse_block_size": 128,
            "sparse_probe_every": 50,
            "sparse_probe_tokens": 256,
        },
        "stability": {
            "qk_norm": bool(qk_norm),
            "ortho_init": bool(ortho_init),
            "spectral_norm": bool(spectral_norm),
        },
        "optim": {"lr": float(lr)},
        "eval": {"long_context_batch_size": 1},
    }


def write_yaml(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")


def run_train_cfg(cfg_path: Path) -> tuple[int, str, str, float]:
    cmd = [sys.executable, "-m", "orion.train", "--config", str(cfg_path), "--device", DEVICE]
    env = os.environ.copy()
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    t0 = time.time()
    proc = subprocess.run(cmd, text=True, capture_output=True, env=env, cwd=Path.cwd())
    dt = time.time() - t0
    return proc.returncode, proc.stdout, proc.stderr, dt


def read_metrics(metrics_path: Path) -> dict:
    if not metrics_path.exists():
        return {}

    step_rows = []
    window_rows = []
    run_rows = []
    with metrics_path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            t = row.get("type")
            if t == "step":
                step_rows.append(row)
            elif t == "window":
                window_rows.append(row)
            elif t == "run":
                run_rows.append(row)

    out = {}
    if step_rows:
        tail = step_rows[-min(20, len(step_rows)) :]
        out["final_loss"] = float(step_rows[-1].get("loss", math.nan))
        out["final_ppl"] = float(step_rows[-1].get("ppl", math.nan))
        out["final_acc_top1"] = float(step_rows[-1].get("accuracy_top1", math.nan))
        out["train_tok_per_s"] = float(
            np.mean([r.get("throughput_tokens_per_sec", math.nan) for r in tail])
        )

    if window_rows:
        w = window_rows[-1]
        out["vram_peak_mib"] = float(w.get("vram_peak_mib", math.nan))
        out["attn_entropy_norm"] = float(w.get("attention_entropy_normalized", math.nan))
        out["valid_neighbors"] = float(w.get("valid_neighbor_fraction", math.nan))
        out["valid_vs_causal_cap"] = float(w.get("valid_neighbor_fraction_vs_causal_cap", math.nan))
        out["window_mass_pct"] = float(w.get("attention_mass_window_pct", math.nan))
        out["expander_mass_pct"] = float(w.get("attention_mass_expander_pct", math.nan))
        out["divergence_rate"] = float(w.get("divergence_rate", math.nan))

    if run_rows:
        r = run_rows[-1]
        out["qk_norm"] = bool(r.get("qk_norm", False))
        out["ortho_init"] = bool(r.get("ortho_init", False))
        out["spectral_norm"] = bool(r.get("spectral_norm", False))

    return out


def minmax_score(series: pd.Series, higher_is_better: bool) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return pd.Series(np.ones(len(series)), index=series.index)
    norm = (s - lo) / (hi - lo)
    return norm if higher_is_better else (1.0 - norm)

In [ ]:
search_rows = []
for d, lr, qk_norm, ortho_init, spectral_norm in itertools.product(
    EXPANDER_DEGREES,
    LR_GRID,
    QK_NORM_OPTIONS,
    ORTHO_INIT_OPTIONS,
    SPECTRAL_NORM_OPTIONS,
):
    # Skip contradictory heavy controls by default if desired (none skipped currently).
    search_rows.append(
        {
            "window_size": WINDOW_SIZE,
            "expander_degree": d,
            "lr": lr,
            "qk_norm": qk_norm,
            "ortho_init": ortho_init,
            "spectral_norm": spectral_norm,
        }
    )

search_df = pd.DataFrame(search_rows).drop_duplicates().reset_index(drop=True)
if MAX_STAGE1_RUNS is not None:
    search_df = search_df.head(int(MAX_STAGE1_RUNS)).copy()

print(f"Stage 1 candidates: {len(search_df)}")
display(search_df.head(20))

In [ ]:
stage1_results = []
batch_size = BATCH_BY_SEQ[SEQ_LEN]

for idx, cand in search_df.iterrows():
    for seed in STAGE1_SEEDS:
        trial_tag = (
            f"s1_idx{idx:03d}_T{SEQ_LEN}_w{int(cand.window_size)}_d{int(cand.expander_degree)}"
            f"_lr{cand.lr}_qk{int(bool(cand.qk_norm))}"
        ).replace(".", "p")
        run_dir = RUN_ROOT / "stage1" / trial_tag
        cfg_path = CFG_ROOT / "stage1" / f"{trial_tag}.yaml"

        cfg = build_sparse_cfg(
            out_dir=run_dir,
            seed=int(seed),
            seq_len=SEQ_LEN,
            batch_size=batch_size,
            lr=float(cand.lr),
            window_size=int(cand.window_size),
            expander_degree=int(cand.expander_degree),
            qk_norm=bool(cand.qk_norm),
            ortho_init=bool(cand.ortho_init),
            spectral_norm=bool(cand.spectral_norm),
            train_tokens_target=STAGE1_TRAIN_TOKENS,
        )
        write_yaml(cfg_path, cfg)

        print(f"[Stage1 {idx + 1}/{len(search_df)}] {trial_tag}")
        rc, stdout, stderr, elapsed_s = run_train_cfg(cfg_path)

        (run_dir / "stdout.log").write_text(stdout, encoding="utf-8")
        (run_dir / "stderr.log").write_text(stderr, encoding="utf-8")

        metrics = read_metrics(run_dir / "metrics.jsonl")
        err_blob = ((stderr or "") + "\n" + (stdout or ""))[-12000:]

        row = {
            "stage": "stage1",
            "trial_tag": trial_tag,
            "status": "ok" if rc == 0 else "failed",
            "returncode": int(rc),
            "elapsed_s": float(elapsed_s),
            "seed": int(seed),
            "seq_len": int(SEQ_LEN),
            "batch_size": int(batch_size),
            "train_tokens_target": int(STAGE1_TRAIN_TOKENS),
            **cand.to_dict(),
            **metrics,
            "error_tail": err_blob if rc != 0 else "",
            "run_dir": str(run_dir),
            "cfg_path": str(cfg_path),
        }
        stage1_results.append(row)

stage1_df = pd.DataFrame(stage1_results)
stage1_csv = RUN_ROOT / "stage1_results.csv"
stage1_df.to_csv(stage1_csv, index=False)
print("Saved:", stage1_csv)
print(stage1_df.status.value_counts(dropna=False).to_string())
display(stage1_df.head(20))

In [ ]:
stage1_ok = stage1_df[stage1_df.status == "ok"].copy()
if len(stage1_ok) == 0:
    raise RuntimeError("No successful Stage 1 runs.")

# Basic feasibility filter
if "valid_vs_causal_cap" in stage1_ok.columns:
    stage1_ok = stage1_ok[
        (stage1_ok.valid_vs_causal_cap.isna()) | (stage1_ok.valid_vs_causal_cap >= 0.95)
    ].copy()

group_cols = ["window_size", "expander_degree", "lr", "qk_norm", "ortho_init", "spectral_norm"]
agg = stage1_ok.groupby(group_cols, as_index=False).agg(
    final_ppl_mean=("final_ppl", "mean"),
    train_tok_per_s_mean=("train_tok_per_s", "mean"),
    vram_peak_mib_mean=("vram_peak_mib", "mean"),
    runs=("trial_tag", "count"),
)

agg["score_quality"] = minmax_score(agg["final_ppl_mean"], higher_is_better=False)
agg["score_speed"] = minmax_score(agg["train_tok_per_s_mean"], higher_is_better=True)
agg["score_vram"] = minmax_score(agg["vram_peak_mib_mean"], higher_is_better=False)

# Quality-first stage-1 rank
agg["stage1_score"] = (
    0.65 * agg["score_quality"] + 0.25 * agg["score_speed"] + 0.10 * agg["score_vram"]
)
agg = agg.sort_values("stage1_score", ascending=False).reset_index(drop=True)

top_candidates = agg.head(min(TOP_K, len(agg))).copy()
top_candidates.to_csv(RUN_ROOT / "stage1_top_candidates.csv", index=False)

print("Top candidates after Stage 1:")
display(top_candidates)

In [ ]:
stage2_results = []
batch_size = BATCH_BY_SEQ[SEQ_LEN]

for idx, cand in top_candidates.iterrows():
    for seed in STAGE2_SEEDS:
        trial_tag = (
            f"s2_rank{idx + 1:02d}_T{SEQ_LEN}_w{int(cand.window_size)}_d{int(cand.expander_degree)}"
            f"_lr{cand.lr}_qk{int(bool(cand.qk_norm))}_seed{seed}"
        ).replace(".", "p")
        run_dir = RUN_ROOT / "stage2" / trial_tag
        cfg_path = CFG_ROOT / "stage2" / f"{trial_tag}.yaml"

        cfg = build_sparse_cfg(
            out_dir=run_dir,
            seed=int(seed),
            seq_len=SEQ_LEN,
            batch_size=batch_size,
            lr=float(cand.lr),
            window_size=int(cand.window_size),
            expander_degree=int(cand.expander_degree),
            qk_norm=bool(cand.qk_norm),
            ortho_init=bool(cand.ortho_init),
            spectral_norm=bool(cand.spectral_norm),
            train_tokens_target=STAGE2_TRAIN_TOKENS,
        )
        write_yaml(cfg_path, cfg)

        print(f"[Stage2 {idx + 1}/{len(top_candidates)}] {trial_tag}")
        rc, stdout, stderr, elapsed_s = run_train_cfg(cfg_path)

        (run_dir / "stdout.log").write_text(stdout, encoding="utf-8")
        (run_dir / "stderr.log").write_text(stderr, encoding="utf-8")

        metrics = read_metrics(run_dir / "metrics.jsonl")
        err_blob = ((stderr or "") + "\n" + (stdout or ""))[-12000:]

        val_loss = math.nan
        val_ppl = math.nan
        val_error = ""
        if rc == 0 and (run_dir / "checkpoint.pt").exists():
            try:
                val_loss, val_ppl = evaluate_val_ppl(
                    cfg, run_dir / "checkpoint.pt", batches=VAL_EVAL_BATCHES_STAGE2
                )
            except Exception as exc:
                val_error = str(exc)

        row = {
            "stage": "stage2",
            "trial_tag": trial_tag,
            "status": "ok" if rc == 0 else "failed",
            "returncode": int(rc),
            "elapsed_s": float(elapsed_s),
            "seed": int(seed),
            "seq_len": int(SEQ_LEN),
            "batch_size": int(batch_size),
            "train_tokens_target": int(STAGE2_TRAIN_TOKENS),
            **{
                k: cand[k]
                for k in [
                    "window_size",
                    "expander_degree",
                    "lr",
                    "qk_norm",
                    "ortho_init",
                    "spectral_norm",
                ]
            },
            **metrics,
            "val_loss": val_loss,
            "val_ppl": val_ppl,
            "val_eval_error": val_error,
            "error_tail": err_blob if rc != 0 else "",
            "run_dir": str(run_dir),
            "cfg_path": str(cfg_path),
        }
        stage2_results.append(row)

stage2_df = pd.DataFrame(stage2_results)
stage2_csv = RUN_ROOT / "stage2_results.csv"
stage2_df.to_csv(stage2_csv, index=False)
print("Saved:", stage2_csv)
print(stage2_df.status.value_counts(dropna=False).to_string())
display(stage2_df.head(20))

In [ ]:
stage2_ok = stage2_df[stage2_df.status == "ok"].copy()
if len(stage2_ok) == 0:
    raise RuntimeError("No successful Stage 2 runs.")

group_cols = ["window_size", "expander_degree", "lr", "qk_norm", "ortho_init", "spectral_norm"]
summary = stage2_ok.groupby(group_cols, as_index=False).agg(
    val_ppl_mean=("val_ppl", "mean"),
    val_ppl_std=("val_ppl", "std"),
    train_tok_per_s_mean=("train_tok_per_s", "mean"),
    vram_peak_mib_mean=("vram_peak_mib", "mean"),
    final_ppl_mean=("final_ppl", "mean"),
    runs=("trial_tag", "count"),
)

# Fallback if val_ppl is missing
quality_col = "val_ppl_mean" if summary["val_ppl_mean"].notna().any() else "final_ppl_mean"

summary["score_quality"] = minmax_score(summary[quality_col], higher_is_better=False)
summary["score_speed"] = minmax_score(summary["train_tok_per_s_mean"], higher_is_better=True)
summary["score_vram"] = minmax_score(summary["vram_peak_mib_mean"], higher_is_better=False)

# Final ranking: quality-first, then speed, then memory
summary["final_score"] = (
    0.60 * summary["score_quality"] + 0.25 * summary["score_speed"] + 0.15 * summary["score_vram"]
)
summary = summary.sort_values("final_score", ascending=False).reset_index(drop=True)

best = summary.iloc[0].to_dict()
summary.to_csv(RUN_ROOT / "stage2_summary_ranked.csv", index=False)

print("Ranked Stage 2 summary:")
display(summary)
print("\nBest sparse combination:")
for k in [
    "window_size",
    "expander_degree",
    "lr",
    "qk_norm",
    "ortho_init",
    "spectral_norm",
    "final_score",
]:
    print(f"{k}: {best.get(k)}")

In [ ]:
best_cfg = build_sparse_cfg(
    out_dir=RUN_ROOT / "best_sparse_final_run",
    seed=int(STAGE2_SEEDS[0]),
    seq_len=int(SEQ_LEN),
    batch_size=int(BATCH_BY_SEQ[SEQ_LEN]),
    lr=float(best["lr"]),
    window_size=int(best["window_size"]),
    expander_degree=int(best["expander_degree"]),
    qk_norm=bool(best["qk_norm"]),
    ortho_init=bool(best["ortho_init"]),
    spectral_norm=bool(best["spectral_norm"]),
    train_tokens_target=int(STAGE2_TRAIN_TOKENS),
)

best_cfg_path = RUN_ROOT / "best_sparse_config.yaml"
write_yaml(best_cfg_path, best_cfg)
print("Saved best config:", best_cfg_path)
print(best_cfg_path.read_text(encoding="utf-8"))

# Optional: also write to reusable variant path in repo.
repo_variant_path = Path("configs/experiments/variants/sparse_tuned_best.yaml")
write_yaml(repo_variant_path, best_cfg)
print("Updated reusable variant:", repo_variant_path)

In [ ]:
if len(stage2_ok) > 0:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=stage2_ok, x="expander_degree", y="val_ppl")
    plt.title("Validation PPL by Expander Degree (Stage 2)")
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=stage2_ok, x="expander_degree", y="train_tok_per_s")
    plt.title("Training Throughput by Expander Degree (Stage 2)")
    plt.show()